# 🌾 AgriSathi AI Advisor — Complete Colab Notebook
## GenAI Project | Domain-Specific RAG + Fine-tuned LLM for Farmers

### ✅ All 7 Mandatory Criteria Covered:
| # | Criterion | Cell |
|---|-----------|------|
| i | Dataset quality, preprocessing, proper split | Step 3–5 |
| ii | PEFT (QLoRA) fine-tuning with justification | Step 7–8 |
| iii | Baseline comparison (3 models) | Step 6, 9 |
| iv | FAISS + SQLite data storage | Step 4, 10 |
| v | BLEU, ROUGE-1/2/L quantitative evaluation | Step 9 |
| vi | Hallucination & error analysis | Step 11 |
| vii | Real-world applicability demo | Step 12 |

> **Runtime:** Runtime → Change runtime type → **T4 GPU**

## ✅ Step 1 — GPU Check

In [ ]:
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')

## ✅ Step 2 — Install Libraries

In [ ]:
!pip install -q unsloth
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers datasets peft trl accelerate bitsandbytes
!pip install -q langchain langchain-community faiss-cpu sentence-transformers
!pip install -q pdfplumber rouge-score nltk bert-score pandas scikit-learn kaggle
print('✅ All libraries installed!')

## ✅ Step 3 — Mount Drive + Setup Folders

In [ ]:
from google.colab import drive
import os, sqlite3

drive.mount('/content/drive')

BASE     = '/content/AgriSathi'
DATA_RAW = f'{BASE}/data/raw'
DATA_PRO = f'{BASE}/data/processed'
DATA_EMB = f'{BASE}/data/embeddings'
MODEL    = f'{BASE}/models'
RESULTS  = f'{BASE}/results'
DRIVE    = '/content/drive/MyDrive/AgriSathi'
DB_PATH  = f'{BASE}/agrisathi.db'

for d in [DATA_RAW,DATA_PRO,DATA_EMB,MODEL,RESULTS,DRIVE,f'{DATA_RAW}/pdfs']:
    os.makedirs(d, exist_ok=True)

# iv. SQLite storage setup
conn = sqlite3.connect(DB_PATH)
conn.executescript('''
    CREATE TABLE IF NOT EXISTS training_runs (
        id INTEGER PRIMARY KEY, run_name TEXT, model TEXT,
        epochs INTEGER, lr REAL, lora_rank INTEGER,
        final_bleu REAL, final_rougeL REAL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS query_logs (
        id INTEGER PRIMARY KEY, question TEXT, model TEXT,
        answer TEXT, bleu REAL, rougeL REAL,
        inference_time REAL, timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS dataset_registry (
        id INTEGER PRIMARY KEY, source TEXT, split TEXT,
        num_samples INTEGER, language TEXT
    );
''')
conn.commit(); conn.close()
print('✅ Folders + SQLite DB ready!')

## ✅ Step 4 — Download Datasets (Kaggle)

In [ ]:
from google.colab import files
print('Upload kaggle.json (from https://www.kaggle.com/settings → API → Create Token)')
uploaded = files.upload()

In [ ]:
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print('Downloading KCC dataset...')
!kaggle datasets download -d mahendrabishnoi2/kisan-call-center-kcc-dataset -p {DATA_RAW} --unzip
print('Downloading Crop Recommendation...')
!kaggle datasets download -d atharvaingle/crop-recommendation-dataset -p {DATA_RAW} --unzip
print('✅ Datasets downloaded!')

## ✅ Step 5 — Preprocessing + EDA + Alpaca Format
### Criterion i: Dataset quality, preprocessing, proper 80/10/10 split

In [ ]:
import pandas as pd, json, re, glob, sqlite3
from sklearn.model_selection import train_test_split

def clean_text(text):
    if not isinstance(text, str): return ''
    return re.sub(r'\s+', ' ', text.strip())

all_records = []

# --- KCC Dataset ---
kcc_files = glob.glob(f'{DATA_RAW}/**/*kcc*', recursive=True) + glob.glob(f'{DATA_RAW}/*KCC*')
print('KCC files:', kcc_files)
if kcc_files:
    df_kcc = pd.read_csv(kcc_files[0], low_memory=False)
    print('KCC shape:', df_kcc.shape)
    print('KCC columns:', df_kcc.columns.tolist())
    q_col = next((c for c in df_kcc.columns if 'query' in c.lower() or 'question' in c.lower()), df_kcc.columns[0])
    a_col = next((c for c in df_kcc.columns if 'answer' in c.lower() or 'response' in c.lower()), df_kcc.columns[1])
    for _, row in df_kcc.dropna(subset=[q_col, a_col]).iterrows():
        q = clean_text(str(row[q_col]))
        a = clean_text(str(row[a_col]))
        if len(q) > 10 and len(a) > 20:
            all_records.append({'instruction': q, 'input': '', 'output': a})
    print(f'KCC records added: {len(all_records)}')

# --- Crop Recommendation Dataset ---
crop_files = glob.glob(f'{DATA_RAW}/*crop*', recursive=True)
if crop_files:
    df_crop = pd.read_csv(crop_files[0])
    for _, row in df_crop.iterrows():
        q = (f"Meri zameen mein N={row.get('N',0)}, P={row.get('P',0)}, K={row.get('K',0)}, "
             f"temperature={row.get('temperature',25):.1f}C, humidity={row.get('humidity',60):.1f}%, "
             f"pH={row.get('ph',7):.1f} hai. Kaun si fasal lagaoon?")
        a = f"{row.get('label','unknown')} fasal ke liye yeh conditions suitable hain."
        all_records.append({'instruction': q, 'input': '', 'output': a})
    print(f'Crop records added: {len(all_records)}')

print(f'\nTotal records: {len(all_records)}')

# --- EDA ---
df_all = pd.DataFrame(all_records)
print('\n=== EDA ===')
print(f'Null values: {df_all.isnull().sum().sum()}')
print(f'Duplicates: {df_all.duplicated().sum()}')
print(f'Avg instruction len: {df_all.instruction.str.len().mean():.0f} chars')
print(f'Avg output len: {df_all.output.str.len().mean():.0f} chars')
df_all = df_all.drop_duplicates(subset=['instruction'])
print(f'After dedup: {len(df_all)} records')

# --- 80/10/10 Split ---
train_df, temp_df = train_test_split(df_all, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f'\nSplit: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}')

train_df.to_csv(f'{DATA_PRO}/train.csv', index=False)
val_df.to_csv(f'{DATA_PRO}/val.csv',   index=False)
test_df.to_csv(f'{DATA_PRO}/test.csv',  index=False)

# Save to SQLite registry
conn = sqlite3.connect(DB_PATH)
for split, df in [('train',train_df),('val',val_df),('test',test_df)]:
    conn.execute('INSERT INTO dataset_registry(source,split,num_samples,language) VALUES (?,?,?,?)',
                 ('KCC+Crop+Govt', split, len(df), 'Hinglish/Hindi/English'))
conn.commit(); conn.close()
print('✅ Data saved + registered in SQLite!')

## ✅ Step 6 — Build FAISS Knowledge Base (Criterion iv)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
train_df = pd.read_csv(f'{DATA_PRO}/train.csv')
docs = [Document(page_content=f"Q: {r.instruction}\nA: {r.output}") for _, r in train_df.iterrows()]
chunks = splitter.split_documents(docs)
print(f'Total chunks: {len(chunks)}')

vectorstore = FAISS.from_documents(chunks, embeddings)
FAISS_PATH = f'{DATA_EMB}/faiss_index'
vectorstore.save_local(FAISS_PATH)
print(f'✅ FAISS index saved: {FAISS_PATH}')

# Quick RAG test
test_q = 'Gehu mein pila pan ka kya ilaj hai?'
results = vectorstore.similarity_search(test_q, k=3)
print(f'\nRAG Test Query: {test_q}')
for i, r in enumerate(results,1):
    print(f'  Chunk {i}: {r.page_content[:100]}...')

## ✅ Step 7 — Baseline 1: Pre-trained Model (No Fine-tuning)
### Criterion iii: Part 1 of 3 model comparison

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME  = 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit'
MAX_SEQ_LEN = 2048

print('Loading base model...')
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True
)
FastLanguageModel.for_inference(base_model)
print('✅ Base model loaded!')

In [ ]:
import torch, time
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs
from tqdm import tqdm
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)

def generate_answer(model, tok, question, use_rag=False, vs=None, max_new=256):
    context = ''
    if use_rag and vs:
        chunks = vs.similarity_search(question, k=3)
        context = '\n'.join([c.page_content for c in chunks])
    
    prompt = f"""<s>[INST] Aap AgriSathi AI ho — ek farming expert jo kisano ki madad karta hai.\n"""
    if context:
        prompt += f"Context:\n{context}\n\n"
    prompt += f"Sawaal: {question} [/INST]"
    
    inputs = tok(prompt, return_tensors='pt').to('cuda')
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new, temperature=0.3,
                                  do_sample=True, pad_token_id=tok.eos_token_id)
    t = round(time.time()-start, 2)
    ans = tok.decode(outputs[0], skip_special_tokens=True).split('[/INST]')[-1].strip()
    return ans, t

def evaluate_model(model, tok, test_df, n=50, model_name='model', use_rag=False, vs=None):
    scorer = rs.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    smooth = SmoothingFunction().method4
    sample = test_df.sample(min(n, len(test_df)), random_state=42)
    rows = []
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc=f'Evaluating {model_name}'):
        pred, t = generate_answer(model, tok, row['instruction'], use_rag, vs)
        ref_tokens  = nltk.word_tokenize(row['output'].lower())
        pred_tokens = nltk.word_tokenize(pred.lower())
        bleu  = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
        rouge = scorer.score(row['output'], pred)
        rows.append({'model':model_name,'question':row['instruction'],
                     'reference':row['output'],'prediction':pred,
                     'bleu':bleu,'rouge1':rouge['rouge1'].fmeasure,
                     'rouge2':rouge['rouge2'].fmeasure,'rougeL':rouge['rougeL'].fmeasure,
                     'inference_time':t})
    df = pd.DataFrame(rows)
    summary = {
        'model':model_name, 'n_samples':len(df),
        'avg_bleu':df.bleu.mean(), 'avg_rouge1':df.rouge1.mean(),
        'avg_rouge2':df.rouge2.mean(), 'avg_rougeL':df.rougeL.mean(),
        'avg_inference_time':df.inference_time.mean()
    }
    return summary, df

test_df = pd.read_csv(f'{DATA_PRO}/test.csv')
baseline_summary, baseline_df = evaluate_model(base_model, tokenizer, test_df, n=50, model_name='Base Mistral-7B')
print('\n📊 Baseline Results:')
print(pd.DataFrame([baseline_summary]).to_string(index=False))

## ✅ Step 8 — Baseline 2: Prompt-Engineered Model
### Criterion iii: 3-way comparison requires this step

In [ ]:
def generate_prompt_eng(model, tok, question, max_new=256):
    """Prompt-engineered baseline: system prompt + few-shot examples"""
    few_shot = """Example 1:
Q: Gehu mein kaunsa fertilizer daalna chahiye?
A: Gehu ke liye buwai ke time DAP 50 kg + urea 30 kg per acre daalo.

Example 2:
Q: PM-KISAN kya hai?
A: PM-KISAN mein registered farmers ko 6000 rupaye/year milte hain.
"""
    prompt = (f"<s>[INST] Aap ek farming expert hain jo Indian farmers ko Hindi/Hinglish mein "
              f"practical advice dete hain. Sirf farming, krishi, fasal, beemari, ya sarkari "
              f"yojana ke sawaalon ka jawab do.\n\n{few_shot}\nQ: {question}\nA: [/INST]")
    inputs = tok(prompt, return_tensors='pt').to('cuda')
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new, temperature=0.3,
                                  do_sample=True, pad_token_id=tok.eos_token_id)
    t = round(time.time()-start, 2)
    ans = tok.decode(outputs[0], skip_special_tokens=True).split('[/INST]')[-1].strip()
    return ans, t

# Evaluate prompt-engineered
sample_50 = test_df.sample(50, random_state=42)
scorer = rs.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method4
rows = []
for _, row in tqdm(sample_50.iterrows(), total=50, desc='Prompt-Engineered'):
    pred, t = generate_prompt_eng(base_model, tokenizer, row['instruction'])
    ref_tokens  = nltk.word_tokenize(row['output'].lower())
    pred_tokens = nltk.word_tokenize(pred.lower())
    bleu  = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
    rouge = scorer.score(row['output'], pred)
    rows.append({'model':'Prompt-Engineered','bleu':bleu,
                 'rouge1':rouge['rouge1'].fmeasure,'rouge2':rouge['rouge2'].fmeasure,
                 'rougeL':rouge['rougeL'].fmeasure,'inference_time':t})

pe_df = pd.DataFrame(rows)
pe_summary = {'model':'Prompt-Engineered','n_samples':50,
              'avg_bleu':pe_df.bleu.mean(),'avg_rouge1':pe_df.rouge1.mean(),
              'avg_rouge2':pe_df.rouge2.mean(),'avg_rougeL':pe_df.rougeL.mean(),
              'avg_inference_time':pe_df.inference_time.mean()}
print('\n📊 Prompt-Engineered Results:')
print(pd.DataFrame([pe_summary]).to_string(index=False))

## ✅ Step 9 — QLoRA Fine-tuning
### Criterion ii: PEFT with LoRA/QLoRA + Justification

**Why QLoRA over full fine-tuning?**
1. **VRAM constraint**: T4 GPU has 15GB. 7B model full FT needs ~56GB (bf16). QLoRA needs ~6GB.
2. **Parameter efficiency**: Only ~41M of 7.24B params are trainable (~0.57%) — prevents catastrophic forgetting.
3. **Speed**: Unsloth makes QLoRA 2-3x faster than standard HuggingFace PEFT.
4. **Quality**: NF4 quantization preserves model quality while reducing memory.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    base_model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

# Format dataset
ALPACA_PROMPT = """Below is an instruction that describes a farming task. Write a response in Hindi/Hinglish that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS = tokenizer.eos_token

def format_prompts(examples):
    texts = []
    for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        texts.append(ALPACA_PROMPT.format(inst, inp, out) + EOS)
    return {'text': texts}

dataset = load_dataset('csv', data_files={'train': f'{DATA_PRO}/train.csv'})['train']
dataset = dataset.map(format_prompts, batched=True)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LEN, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=2, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=50, optim='adamw_8bit',
        output_dir=f'{RESULTS}/checkpoints', save_strategy='epoch',
    )
)

print('\n🚀 Starting QLoRA fine-tuning...')
trainer_stats = trainer.train()
print('✅ Fine-tuning complete!')
print(f'Total steps: {trainer_stats.global_step}')
print(f'Training time: {trainer_stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

logs = trainer.state.log_history
steps = [l['step'] for l in logs if 'loss' in l]
losses = [l['loss'] for l in logs if 'loss' in l]

plt.figure(figsize=(8,4))
plt.plot(steps, losses, color='#2D6A4F', linewidth=2)
plt.xlabel('Step'); plt.ylabel('Training Loss')
plt.title('AgriSathi QLoRA Training Loss Curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS}/training_loss.png', dpi=150)
plt.show()
print('✅ Loss curve saved!')

## ✅ Step 10 — Save Model + Log to SQLite

In [ ]:
import shutil, sqlite3

SAVE_PATH  = f'{MODEL}/agrisathi-finetuned'
DRIVE_PATH = f'{DRIVE}/agrisathi-finetuned'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
shutil.copytree(SAVE_PATH, DRIVE_PATH, dirs_exist_ok=True)
shutil.copy(f'{DATA_EMB}/faiss_index', f'{DRIVE}/faiss_index') if os.path.isfile(f'{DATA_EMB}/faiss_index') else None
print(f'✅ Model saved: {SAVE_PATH}')
print(f'✅ Backed up to Drive: {DRIVE_PATH}')

## ✅ Step 11 — Evaluate Fine-tuned + 3-way Comparison
### Criteria iii + v: Baseline comparison + BLEU/ROUGE

In [ ]:
FastLanguageModel.for_inference(model)

# Load FAISS for RAG
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.load_local(f'{DATA_EMB}/faiss_index', embeddings, allow_dangerous_deserialization=True)

# Evaluate fine-tuned (with RAG)
finetuned_summary, finetuned_df = evaluate_model(
    model, tokenizer, test_df, n=50,
    model_name='AgriSathi QLoRA', use_rag=True, vs=vectorstore
)

# 3-way comparison table
comparison = pd.DataFrame([baseline_summary, pe_summary, finetuned_summary])
comparison.to_csv(f'{RESULTS}/model_comparison.csv', index=False)
shutil.copy(f'{RESULTS}/model_comparison.csv', f'{DRIVE}/model_comparison.csv')

print('\n' + '='*65)
print('📊  3-WAY MODEL COMPARISON — AgriSathi')
print('='*65)
print(comparison[['model','avg_bleu','avg_rouge1','avg_rouge2','avg_rougeL','avg_inference_time']].to_string(index=False))

b0 = baseline_summary['avg_bleu']; bft = finetuned_summary['avg_bleu']
r0 = baseline_summary['avg_rougeL']; rft = finetuned_summary['avg_rougeL']
print(f'\n  BLEU improvement (base→finetuned):    +{(bft-b0)/b0*100:.0f}%')
print(f'  ROUGE-L improvement (base→finetuned): +{(rft-r0)/r0*100:.0f}%')

# Log to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute('INSERT INTO training_runs(run_name,model,epochs,lr,lora_rank,final_bleu,final_rougeL) VALUES (?,?,?,?,?,?,?)',
             ('run_001','Mistral-7B QLoRA',2,0.0002,16,finetuned_summary['avg_bleu'],finetuned_summary['avg_rougeL']))
conn.commit(); conn.close()
print('✅ Results logged to SQLite!')

## ✅ Step 12 — Hallucination & Error Analysis
### Criterion vi: Structured failure case analysis

In [ ]:
import json, sqlite3

# Known-answer test cases (ground truth available)
known_cases = [
    {'question': 'Kya gehu ki fasal august mein lagayi ja sakti hai?',
     'ground_truth': 'Nahi. Gehu Rabi fasal hai — October-November mein lagao.',
     'error_type': 'Factual Hallucination'},
    {'question': 'PM-KISAN mein kitna paisa milta hai?',
     'ground_truth': '6000 rupaye per year, teen kiston mein (2000 rupaye har 4 mahine).',
     'error_type': 'Incomplete Answer'},
    {'question': 'Chawal mein blast disease ka ilaj batao',
     'ground_truth': 'Tricyclazole 75% WP 0.6g/litre spray karo.',
     'error_type': 'Vague / Generic'},
    {'question': 'Drip irrigation mein kitna paani bachta hai?',
     'ground_truth': '40-50% paani bachta hai compared to flood irrigation.',
     'error_type': 'Factual Accuracy'},
    {'question': 'Urea mein nitrogen ka percentage kitna hota hai?',
     'ground_truth': 'Urea mein 46% nitrogen hota hai.',
     'error_type': 'Factual Accuracy'},
]

print('\n' + '='*65)
print('🔍 HALLUCINATION & ERROR ANALYSIS')
print('='*65)

results = []
for c in known_cases:
    base_ans,  _ = generate_answer(base_model, tokenizer, c['question'])
    ft_ans,    _ = generate_answer(model, tokenizer, c['question'], use_rag=True, vs=vectorstore)
    
    scorer_r = rs.RougeScorer(['rougeL'], use_stemmer=True)
    base_score = scorer_r.score(c['ground_truth'], base_ans)['rougeL'].fmeasure
    ft_score   = scorer_r.score(c['ground_truth'], ft_ans)['rougeL'].fmeasure
    
    print(f'\n[{c["error_type"]}]')
    print(f'Q: {c["question"]}')
    print(f'Ground Truth: {c["ground_truth"]}')
    print(f'Base  (RL={base_score:.3f}): {base_ans[:120]}...')
    print(f'FT    (RL={ft_score:.3f}): {ft_ans[:120]}...')
    
    results.append({**c, 'base_answer': base_ans, 'ft_answer': ft_ans,
                    'base_rougeL': base_score, 'ft_rougeL': ft_score,
                    'ft_better': ft_score > base_score})

with open(f'{RESULTS}/hallucination_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

wins = sum(r['ft_better'] for r in results)
print(f'\nSummary: Fine-tuned better in {wins}/{len(results)} cases')
print('✅ Saved: results/hallucination_analysis.json')

## ✅ Step 13 — Gradio UI Demo
### Desired: Frontend UI Integration

In [ ]:
!pip install -q gradio
import gradio as gr

def agrisathi_chat(question, model_choice, use_rag):
    m = model if model_choice == 'Fine-tuned QLoRA' else base_model
    ans, t = generate_answer(m, tokenizer, question, use_rag=use_rag, vs=vectorstore if use_rag else None)
    return ans, f'Inference: {t}s'

demo = gr.Interface(
    fn=agrisathi_chat,
    inputs=[
        gr.Textbox(label='Sawaal (Hindi/Hinglish)', placeholder='Gehu mein pila pan kyu aata hai?', lines=2),
        gr.Dropdown(['Fine-tuned QLoRA','Base Mistral-7B'], label='Model', value='Fine-tuned QLoRA'),
        gr.Checkbox(label='Use RAG (FAISS)', value=True)
    ],
    outputs=[
        gr.Textbox(label='AgriSathi Answer', lines=6),
        gr.Textbox(label='Stats')
    ],
    title='🌾 AgriSathi AI Advisor',
    description='Domain-specific farming AI — Fine-tuned Mistral-7B with RAG',
    examples=[
        ['Mere gehu mein pila pan aa raha hai', 'Fine-tuned QLoRA', True],
        ['PM-KISAN mein register kaise karein?', 'Fine-tuned QLoRA', True],
        ['Chawal mein blast disease ka ilaj', 'Fine-tuned QLoRA', True],
    ]
)
demo.launch(share=True)

## ✅ All Done! 🎉
| Step | Task | Criterion |
|------|------|----------|
| 1-2 | GPU + Install | Setup |
| 3 | Drive + SQLite setup | iv |
| 4-5 | Download + Preprocess + EDA + Split | i |
| 6 | FAISS Knowledge Base | iv |
| 7 | Base model baseline | iii |
| 8 | Prompt-engineered baseline | iii |
| 9 | QLoRA Fine-tuning | ii |
| 10 | Save + SQLite logging | iv |
| 11 | 3-way comparison + BLEU/ROUGE | iii + v |
| 12 | Hallucination analysis | vi |
| 13 | Gradio UI demo | Desired |

**Next:** Download model from Drive → put in `models/agrisathi-finetuned/` → run FastAPI backend → open `frontend/dashboard/index.html`